# 031 — Métodos Monte Carlo y simulación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** Valores ≥ 5: posiciones con 5 o 6 → `[6,6,5,6,6,5,6,6]` = 8 de 20 → `Ê = 0.40`. Exacto: `1/3 ≈ 0.333`. Error 0.067; el error estándar teórico es `√(p(1−p)/20) ≈ 0.105`, así que 0.40 está a menos de 1 desviación: perfectamente normal para N=20 — no evidencia de dado cargado.

**E2.** Error ∝ 1/√N. Para la mitad (0.025): ×4 → `N = 1600`. Para un décimo (0.005): ×100 → `N = 40 000`. La precisión Monte Carlo se paga cuadráticamente.

**E3.** Con semilla 31 se obtiene típicamente error ~1e-2 con N=1 000 y ~5e-3 o menos con N=100 000 (los valores exactos dependen del generador, por eso se declara la semilla). La razón de errores debe rondar `√100 = 10` *en promedio* — una corrida individual puede desviarse.

**E4.** Se conservan en esperanza `10 000 × 0.02 = 200` muestras: el 98 % del cómputo se tira. La alternativa es el muestreo por importancia (ponderar en vez de descartar) o MCMC, que muestrean directamente la región compatible con la evidencia.


In [ ]:
result = run_lab("probability", seed=31)
assert result["kind"] == "probability"
assert result["evidence"]
show(result)


In [ ]:
import random, math
muestras = [3,1,4,6,6,2,5,3,6,1,2,4,6,5,3,6,1,2,6,4]
p_hat = sum(1 for m in muestras if m >= 5)/len(muestras)
se = math.sqrt(p_hat*(1-p_hat)/len(muestras))
print(f"E1: {p_hat:.3f} (exacto 0.333, error estandar {se:.3f})")

random.seed(31)
for N in (1_000, 100_000):
    dentro = sum(1 for _ in range(N) if random.random()**2 + random.random()**2 <= 1)
    pi_hat = 4*dentro/N
    print(f"E3 N={N}: pi_hat={pi_hat:.5f} error={abs(pi_hat-math.pi):.5f}")
print("E4: se conservan ~", 10_000*0.02, "muestras")


## Reflexión

1. Si ejecutas el laboratorio con dos semillas y los resultados difieren, ¿es un bug? ¿Qué tendría que ser verdad para que la diferencia sí indicara un problema?
2. ¿Por qué el muestreo por rechazo se degrada catastróficamente cuando la evidencia es improbable? ¿Cuántas muestras esperas descartar si `P(e) = 0.001`?
3. Un estimador Monte Carlo dio 0.503 con N=100. ¿Qué necesitas reportar junto al número para que otra persona juzgue si "≈0.5" es una conclusión válida?
